# Conquer Mobile Assets — Visual Inspection & Godot 4.3 Import Validation (One Cell)

مرحلة مراجعة بصرية فعلية للـ174 مجسمًا والـ90 صورة/خامة، ثم اختبار استيراد عملي باستخدام Godot 4.3 داخل مشروع اختبار مؤقت ومعزول. تنشئ معاينات وContact Sheets وتقارير استيراد وسجل أخطاء، وترفع النتائج إلى نفس GitHub Release.

**خلية كود واحدة فقط. CPU عادي، لا GPU ولا Drive ولا إدخال مسارات.** تستخدم Secret `GITHUB_TOKEN` الموجود بالفعل. الاختبار معزول ولا يغيّر مشروع «صراع الأساطير». لا يعتبر نجاح الاستيراد إثباتًا لجودة الخامات بصريًا أو جاهزية الأنيميشن أو اختبار اللعب الفعلي.

In [ ]:
%pip -q install requests pillow trimesh matplotlib numpy

import os, json, csv, io, zipfile, hashlib, shutil, subprocess, stat, requests, re, math, time
from pathlib import Path, PurePosixPath
from collections import Counter
from PIL import Image, ImageDraw, ImageFont
import numpy as np
import trimesh
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from google.colab import userdata
from IPython.display import display, Markdown, Image as IPImage

OWNER="ZzwahwahzZ"; SOURCE_REPO="Conquer-Mobile-Assets."; TAG="mobile-extracted-assets"
WORK=Path("/content/conquer_mobile_visual_validation")
ARCHIVE=WORK/"conquer-mobile-prepared-assets.zip"; ROOT=WORK/"prepared"; REPORTS=WORK/"reports"; PREVIEWS=WORK/"model-previews"
API=f"https://api.github.com/repos/{OWNER}/{SOURCE_REPO}"
WORK.mkdir(parents=True,exist_ok=True); shutil.rmtree(ROOT,ignore_errors=True)
ROOT.mkdir(parents=True); REPORTS.mkdir(parents=True); PREVIEWS.mkdir(parents=True)
TOKEN=userdata.get("GITHUB_TOKEN")
if not TOKEN: raise RuntimeError("Colab Secret GITHUB_TOKEN is missing.")
H={"Authorization":f"Bearer {TOKEN}","Accept":"application/vnd.github+json","X-GitHub-Api-Version":"2022-11-28"}
print("Resolving prepared asset package from fixed GitHub Release...")
rel=requests.get(f"{API}/releases/tags/{TAG}",headers=H,timeout=60); rel.raise_for_status(); release=rel.json()
assets={a["name"]:a for a in release.get("assets",[])}
required=["conquer-mobile-prepared-assets.zip","asset-preparation-manifest.json","asset-preparation-manifest.csv"]
missing=[n for n in required if n not in assets]
if missing: raise RuntimeError("Missing release assets: "+", ".join(missing))
for name in required:
    a=assets[name]; dest=WORK/name
    r=requests.get(a["browser_download_url"],headers={**H,"Accept":"application/octet-stream"},timeout=300)
    r.raise_for_status(); dest.write_bytes(r.content)
    if len(r.content)!=a["size"]: raise RuntimeError(f"Size mismatch for {name}")
    print("DOWNLOADED + SIZE VERIFIED",name,len(r.content))
shutil.copy2(WORK/required[1],REPORTS/required[1]); shutil.copy2(WORK/required[2],REPORTS/required[2])
with zipfile.ZipFile(ARCHIVE) as z:
    bad=z.testzip()
    if bad: raise RuntimeError("ZIP CRC failure: "+bad)
    for info in z.infolist():
        p=PurePosixPath(info.filename.replace("\\","/"))
        if p.is_absolute() or ".." in p.parts or stat.S_ISLNK((info.external_attr>>16)&0xffff):
            raise RuntimeError("Unsafe archive entry: "+info.filename)
    z.extractall(ROOT)
manifest=json.loads((REPORTS/required[1]).read_text(encoding="utf-8"))
model_files=sorted(p for p in ROOT.rglob("*") if p.is_file() and p.suffix.lower() in {".glb",".gltf",".obj",".fbx",".dae",".blend",".3ds",".stl",".ply"})
image_files=sorted(p for p in ROOT.rglob("*") if p.is_file() and p.suffix.lower() in {".png",".jpg",".jpeg",".webp",".tga",".bmp",".dds",".tif",".tiff"})
print(f"EXTRACTED PREPARED PACKAGE | models={len(model_files)} | images/textures={len(image_files)}")
if len(model_files)!=174: print("WARNING: prepared model count differs from expected 174")
if len(image_files)!=90: print("WARNING: prepared image count differs from expected 90")

# 1) Visual image/texture contact sheet (thumbnails only; original files remain untouched).
def contact_sheet(files,out,title,cols=6,cell=(180,145)):
    rows=max(1,math.ceil(len(files)/cols)); sheet=Image.new("RGB",(cols*cell[0],rows*cell[1]),(32,34,40)); d=ImageDraw.Draw(sheet)
    for i,p in enumerate(files):
        x=(i%cols)*cell[0]; y=(i//cols)*cell[1]
        try:
            im=Image.open(p); im.thumbnail((cell[0]-12,cell[1]-34))
            if im.mode in ("RGBA","LA") or "transparency" in im.info:
                bg=Image.new("RGBA",im.size,(65,65,65,255)); bg.alpha_composite(im.convert("RGBA")); im=bg.convert("RGB")
            else: im=im.convert("RGB")
            sheet.paste(im,(x+(cell[0]-im.width)//2,y+4))
            label=p.name[:24]
            d.text((x+5,y+cell[1]-25),label,fill=(245,245,245))
        except Exception as e: d.text((x+5,y+10),"UNREADABLE: "+p.suffix,fill=(255,160,120))
    sheet.save(out,quality=88); return out
img_sheet=contact_sheet(image_files,REPORTS/"texture-image-contact-sheet.jpg","Textures")
print("TEXTURE CONTACT SHEET CREATED",len(image_files))

# 2) Render static model previews with CPU only.
def render_mesh(path,out):
    try:
        loaded=trimesh.load(path,force="scene",process=False)
        geoms=[]
        if isinstance(loaded,trimesh.Trimesh): geoms=[loaded]
        elif isinstance(loaded,trimesh.Scene): geoms=[g for g in loaded.geometry.values() if isinstance(g,trimesh.Trimesh)]
        geoms=[g for g in geoms if len(g.vertices)>0 and len(g.faces)>0]
        if not geoms: return "no_mesh_geometry"
        fig=plt.figure(figsize=(3.4,3.4),dpi=85); ax=fig.add_subplot(111,projection="3d")
        allv=[]
        for g in geoms[:12]:
            v=np.asarray(g.vertices); f=np.asarray(g.faces)
            if len(f)>2500: f=f[np.linspace(0,len(f)-1,2500,dtype=int)]
            ax.add_collection3d(Poly3DCollection(v[f],linewidths=0.03,edgecolors="none",alpha=0.95))
            allv.append(v)
        vv=np.vstack(allv); lo=vv.min(axis=0); hi=vv.max(axis=0); center=(lo+hi)/2; span=max(float((hi-lo).max()),0.001)/2
        ax.set_xlim(center[0]-span,center[0]+span); ax.set_ylim(center[1]-span,center[1]+span); ax.set_zlim(center[2]-span,center[2]+span)
        ax.view_init(elev=22,azim=35); ax.set_axis_off(); fig.tight_layout(pad=0)
        fig.savefig(out,transparent=False,bbox_inches="tight",pad_inches=0.02); plt.close(fig)
        return "rendered"
    except Exception as e:
        return "render_error: "+str(e)[:180]
preview_rows=[]
for i,p in enumerate(model_files,1):
    out=PREVIEWS/(re.sub(r"[^A-Za-z0-9_.-]+","_",p.stem)[:60]+f"_{i:03d}.png")
    status=render_mesh(p,out)
    preview_rows.append({"path":str(p.relative_to(ROOT)),"preview":str(out.relative_to(WORK)) if status=="rendered" else "","status":status})
    if i%20==0 or i==len(model_files): print(f"VISUAL PREVIEWS {i}/{len(model_files)}")
preview_sheet=contact_sheet([WORK/r["preview"] for r in preview_rows if r["preview"]],REPORTS/"3d-model-visual-contact-sheet.jpg", "Models",cols=5,cell=(210,210))
with open(REPORTS/"visual-preview-index.csv","w",newline="",encoding="utf-8-sig") as f:
    w=csv.DictWriter(f,fieldnames=["path","preview","status"]); w.writeheader(); w.writerows(preview_rows)
print("3D PREVIEW SUMMARY",dict(Counter(r["status"].split(":")[0] for r in preview_rows)))

# 3) Practical Godot 4.3 headless import test in an isolated scratch project.
# This does not modify the user's actual game project.
GODOT_DIR=WORK/"godot"; GODOT_DIR.mkdir(exist_ok=True)
godot_zip=GODOT_DIR/"godot-4.3-linux.zip"
godot_url="https://github.com/godotengine/godot/releases/download/4.3-stable/Godot_v4.3-stable_linux.x86_64.zip"
print("Downloading official Godot 4.3 headless-compatible editor binary...")
gr=requests.get(godot_url,stream=True,timeout=300); gr.raise_for_status()
with open(godot_zip,"wb") as f:
    for chunk in gr.iter_content(8*1024*1024):
        if chunk: f.write(chunk)
with zipfile.ZipFile(godot_zip) as z: z.extractall(GODOT_DIR)
binaries=list(GODOT_DIR.glob("Godot_v4.3-stable_linux.x86_64"))
if not binaries: raise RuntimeError("Godot 4.3 binary not found in official archive.")
godot=binaries[0]; godot.chmod(0o755)
TEST=WORK/"godot_import_sandbox"; shutil.rmtree(TEST,ignore_errors=True)
(TEST/"assets").mkdir(parents=True)
# Copy model files with adjacent prepared textures while preserving their prepared relative paths.
for p in model_files:
    relp=p.relative_to(ROOT); dest=TEST/"assets"/relp; dest.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(p,dest)
for p in image_files:
    relp=p.relative_to(ROOT); dest=TEST/"assets"/relp; dest.parent.mkdir(parents=True,exist_ok=True)
    if not dest.exists(): shutil.copy2(p,dest)
(TEST/"project.godot").write_text('config_version=5\n\n[application]\nconfig/name="Asset Import Validation Sandbox"\n\n[rendering]\nrenderer/rendering_method="gl_compatibility"\n',encoding="utf-8")
cmd=[str(godot),"--headless","--editor","--path",str(TEST),"--import","--quit"]
proc=subprocess.run(cmd,capture_output=True,text=True,timeout=900)
log=(proc.stdout or "")+"\n"+(proc.stderr or "")
(REPORTS/"godot-4.3-import-log.txt").write_text(log,encoding="utf-8")
imported=list((TEST/".godot"/"imported").glob("*")) if (TEST/".godot"/"imported").exists() else []
errors=[line for line in log.splitlines() if re.search(r"ERROR|Parse Error|Failed loading|Import failed|Can't load",line,re.I)]
test_summary={
 "godot_version":"4.3-stable",
 "exit_code":proc.returncode,
 "model_files_submitted":len(model_files),
 "image_texture_files_submitted":len(image_files),
 "imported_cache_files":len(imported),
 "error_lines":errors[:100],
 "result":"PASS" if proc.returncode==0 and not errors else "REVIEW_REQUIRED",
 "scope":"isolated scratch project only; no actual Clash of Legends project files modified",
 "runtime_gameplay_test":"NOT_PERFORMED",
 "animation_rigging_test":"NOT_PERFORMED"
}
(REPORTS/"godot-4.3-import-report.json").write_text(json.dumps(test_summary,ensure_ascii=False,indent=2),encoding="utf-8")
print("GODOT IMPORT TEST",json.dumps(test_summary,ensure_ascii=False))
# Package all review outputs and upload them to the same release.
review_zip=WORK/"conquer-mobile-visual-validation.zip"
with zipfile.ZipFile(review_zip,"w",zipfile.ZIP_DEFLATED,compresslevel=4) as z:
    for p in list(REPORTS.iterdir())+list(PREVIEWS.iterdir()):
        if p.is_file(): z.write(p,p.relative_to(WORK))
upload_names=["conquer-mobile-visual-validation.zip","3d-model-visual-contact-sheet.jpg","texture-image-contact-sheet.jpg","visual-preview-index.csv","godot-4.3-import-report.json","godot-4.3-import-log.txt"]
upload_paths=[review_zip,REPORTS/"3d-model-visual-contact-sheet.jpg",REPORTS/"texture-image-contact-sheet.jpg",REPORTS/"visual-preview-index.csv",REPORTS/"godot-4.3-import-report.json",REPORTS/"godot-4.3-import-log.txt"]
for name,p in zip(upload_names,upload_paths):
    old=assets.get(name)
    if old:
        d=requests.delete(old["url"],headers=H,timeout=60); d.raise_for_status()
    with open(p,"rb") as f:
        u=requests.post(release["upload_url"].split("{",1)[0],params={"name":name},headers={**H,"Content-Type":"application/octet-stream"},data=f,timeout=600)
    if u.status_code not in (200,201): raise RuntimeError(f"Upload failed {name}: {u.status_code} {u.text[:500]}")
    chk=requests.get(f"{API}/releases/tags/{TAG}",headers=H,timeout=60); chk.raise_for_status()
    found={a["name"]:a for a in chk.json().get("assets",[])}
    if name not in found or found[name]["size"]!=p.stat().st_size: raise RuntimeError("Upload verification failed: "+name)
    print("UPLOADED + VERIFIED",name,p.stat().st_size)
print("VISUAL + PRACTICAL VALIDATION PIPELINE COMPLETE")
print("Release:",release["html_url"])
display(Markdown("### Visual review outputs"))
display(IPImage(filename=str(REPORTS/"3d-model-visual-contact-sheet.jpg")))
display(IPImage(filename=str(REPORTS/"texture-image-contact-sheet.jpg")))
